In [1]:
import pandas as pd
import janitor

pd.set_option("display.max_columns", 120)

In [2]:
df_pkg = pd.read_csv("../input/pkg_human_downloads.csv").drop_duplicates(
    "pkg", ignore_index=True
)
df_pkg.head(3)

,pkg,date,download_count,tt_downloads,treatment,boughtstars,treatment2,diff_intervention,treated
0,a-pandas-ex-df2htmlstring,2023-04-25,0.0,0.0,0,0,0,0.0,0.0
1,aa-simplewiki,2023-04-25,17.0,17.0,0,0,0,0.0,0.0
2,aacommpy,2023-04-25,12.0,12.0,1,0,1,18.0,0.0


In [8]:
df_repo = pd.read_csv("../../baltest/input/repo_baselines.csv").select_columns(
    ["html_url", "size_mb", "full_name"]
)
print(df_repo.info())
df_repo.head(3)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 582 entries, 0 to 581
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   html_url   582 non-null    object 
 1   size_mb    582 non-null    float64
 2   full_name  582 non-null    object 
dtypes: float64(1), object(2)
memory usage: 13.8+ KB
None


/home/lsys/social_proof_stars/github_exp/venv_socialproof/lib/python3.10/site-packages/pandas_flavor/register.py:161: FutureWarning: This function will be deprecated in a 1.x release. Please use `jn.select` instead.
  return method(self._obj, *args, **kwargs)


,html_url,size_mb,full_name
0,https://github.com/renanmoretto/ezfinpy,0.016602,renanmoretto/ezfinpy
1,https://github.com/dingyizhao/statplot,0.002930,dingyizhao/statplot
2,https://github.com/deepghs/imgutils,168.523438,deepghs/imgutils


In [9]:
df_check = pd.read_csv("../../get_baseline_profile/input/check-github-url.csv")
df_check.head(3)

,pkg,return_code,github_url,homepage,earliest_release,gh_url_check,github
0,gpt-review,200.0,https://github.com/dciborow/action-gpt/issues,NaN,2023-04-25T22:54:54,0.0,NaN
1,pixivspidercreatedbyhanxu,200.0,NaN,https://gitee.com/UnderTurrets/pixiv-spider,2023-04-23T13:32:51,0.0,NaN
2,text-content-generator,200.0,https://github.com/mheshze/TextContentGenerati...,NaN,2023-04-24T08:11:28,NaN,NaN


In [12]:
df_gharchive_pretreat = (
    pd.read_stata("../../baltest/output/repo_gharchive_events.dta")
    .set_index("full_name")
    .filter(like="_pre")
    .reset_index()
)
df_gharchive_pretreat.head(3)

,full_name,forkevent_pre,pullrequestevent_pre,pushevent_pre,releaseevent_pre,watchevent_pre,issues_opened_pre,issues_closed_pre
0,renanmoretto/ezfinpy,0,0,10,0,1,0,0
1,dingyizhao/statplot,0,0,1,0,0,0,0
2,deepghs/imgutils,0,26,232,2,1,0,0


In [20]:
df_merged = (
    df_pkg.merge(df_check, how="left", on="pkg", validate="1:1")
    # retrieve gh url
    .assign(github_url_lower=lambda df_: df_["github_url"].str.lower())
    # merge by gh url to get repo name
    .merge(
        df_repo.assign(html_url_lower=lambda df_: df_["html_url"].str.lower()),
        how="left",
        left_on="github_url_lower",
        right_on="html_url_lower",
        validate="m:1",
    )
    # merge to get gharchive events
    .merge(
        df_gharchive_pretreat, how="left", on="full_name", validate="m:1"
    ).select_columns(
        [
            "pkg",
            "size_mb",
            "forkevent_pre",
            "pullrequestevent_pre",
            "pushevent_pre",
            "releaseevent_pre",
            "watchevent_pre",
            "issues_opened_pre",
            "issues_closed_pre",
        ]
    )
)
df_merged.to_csv("../output/pkg_repo_size_complexity.csv", index=False)
df_merged

/home/lsys/social_proof_stars/github_exp/venv_socialproof/lib/python3.10/site-packages/pandas_flavor/register.py:161: FutureWarning: This function will be deprecated in a 1.x release. Please use `jn.select` instead.
  return method(self._obj, *args, **kwargs)


,pkg,size_mb,forkevent_pre,pullrequestevent_pre,pushevent_pre,releaseevent_pre,watchevent_pre,issues_opened_pre,issues_closed_pre
0,a-pandas-ex-df2htmlstring,0.004883,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1,aa-simplewiki,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,aacommpy,2.072266,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,aamaze,0.102539,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,acmen,5.543945,1.0,0.0,9.0,1.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...
617,yt5,0.511719,1.0,8.0,31.0,3.0,8.0,0.0,0.0
618,ytchat,29.263672,0.0,22.0,79.0,22.0,0.0,0.0,0.0
619,yze,0.041016,0.0,0.0,29.0,0.0,0.0,0.0,0.0
620,zendron,102.594727,0.0,0.0,181.0,33.0,8.0,0.0,0.0
